In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/laureneproctor/Olympiad-AI.git
%cd Olympiad-AI

Cloning into 'Olympiad-AI'...
remote: Enumerating objects: 1349, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 1349 (delta 83), reused 11 (delta 11), pack-reused 1231 (from 3)
Receiving objects: 100% (1349/1349), 1.47 MiB | 13.01 MiB/s, done.
Resolving deltas: 100% (803/803), done.
/content/Olympiad-AI


In [ ]:
import os
import json
import yaml
import aimo3.scripts.evaluate as evaluate

# all models to compare
MODELS = {
    "baseline_sft_qwen_exp2": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/baseline_sft_qwen_exp2",
        "model_key": "qwen",
    },
    "baseline_sft_deepseekmath_exp2": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/baseline_sft_deepseekmath_exp2",
        "model_key": "deepseekmath",
    },
    "baseline_grpo_qwen_exp2": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/baseline_grpo_qwen_exp2",
        "model_key": "qwen",
    },
    "baseline_grpo_deepseekmath_exp2": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/baseline_grpo_deepseekmath_exp2",
        "model_key": "deepseekmath",
    },
    "sft_qwen_exp1": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/sft_qwen_exp1",
        "model_key": "qwen",
    },
    "sft_deepseekmath_exp1": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/sft_deepseekmath_exp1",
        "model_key": "deepseekmath",
    },
    "sft_deepseekmath_exp3": {
        "checkpoint": "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/sft_deepseekmath_exp3",
        "model_key": "deepseekmath",
    },
}

# Match the YAML defaults
DATA_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/data.yaml"
SFT_CONFIG_PATH = "/content/Olympiad-AI/aimo3/configs/sft.yaml"
DATASET_PATH = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000"
OUTPUT_ROOT = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/runs"

for name, info in MODELS.items():
    eval_cfg = {
        "run": {
            "seed": 42,
        },
        "paths": {
            "data_config_path": DATA_CONFIG_PATH,
            "sft_config_path": SFT_CONFIG_PATH,
            "model_checkpoint": info["checkpoint"],
            "dataset_path": DATASET_PATH,
            "output_dir": os.path.join(OUTPUT_ROOT, f"evaluation_{name}"),
        },
        "evaluation": {
            "split": "test",
            "max_items": 50,
            "eval_pass1": True,
            "pass1_max_new_tokens": 512,
            "eval_majn": True,
            "majn_n_samples": 8,
            "majn_max_items": 50,
            "majn_max_new_tokens": 512,
            "majn_temperature": 0.7,
            "majn_top_p": 0.95,
        },
        "reporting": {
            "save_report": True,
            "report_filename": f"evaluation_report_{name}.json",
            "verbose": True,
        },
    }

    cfg_path = f"/content/Olympiad-AI/aimo3/configs/_tmp_eval_{name}.yaml"
    with open(cfg_path, "w") as f:
        yaml.safe_dump(eval_cfg, f, sort_keys=False)

    print(f"\n=== {name} ===")
    results = evaluate.run_evaluation(cfg_path)
    print(json.dumps(results, indent=2))